In [2]:
import requests
# from bs4 import BeautifulSoup
import re
import time
import os
import spacy
# from fastembed import TextEmbedding

TypeError: ForwardRef._evaluate() missing 1 required keyword-only argument: 'recursive_guard'

In [4]:
article_id = 'Harry_Potter'

In [5]:
MAX_SENTENCE_WORDS = 60
MIN_SENTENCE_WORDS = 10

In [6]:
# model = TextEmbedding(model_name='sentence-transformers/all-MiniLM-L6-v2')
model = TextEmbedding(model_name='BAAI/bge-small-en-v1.5')

_ = list(model.embed(["warmup sentence"]))

In [7]:
TextEmbedding.list_supported_models()

[{'model': 'BAAI/bge-base-en',
  'sources': {'hf': 'Qdrant/fast-bge-base-en',
   'url': 'https://storage.googleapis.com/qdrant-fastembed/fast-bge-base-en.tar.gz',
   '_deprecated_tar_struct': True},
  'model_file': 'model_optimized.onnx',
  'description': 'Text embeddings, Unimodal (text), English, 512 input tokens truncation, Prefixes for queries/documents: necessary, 2023 year.',
  'license': 'mit',
  'size_in_GB': 0.42,
  'additional_files': [],
  'dim': 768,
  'tasks': {}},
 {'model': 'BAAI/bge-base-en-v1.5',
  'sources': {'hf': 'qdrant/bge-base-en-v1.5-onnx-q',
   'url': 'https://storage.googleapis.com/qdrant-fastembed/fast-bge-base-en-v1.5.tar.gz',
   '_deprecated_tar_struct': True},
  'model_file': 'model_optimized.onnx',
  'description': 'Text embeddings, Unimodal (text), English, 512 input tokens truncation, Prefixes for queries/documents: not so necessary, 2023 year.',
  'license': 'mit',
  'size_in_GB': 0.21,
  'additional_files': [],
  'dim': 768,
  'tasks': {}},
 {'model':

In [ ]:
def parse_sentence(sentence, sentences, buf, min):
    if sentence.strip() == '':
        return
    sentence = re.sub(r"\s\s+", ' ', sentence).strip()
    if not sentence.endswith('.'):
        sentence = sentence + '.'

    if buf:
        sentence = ' '.join(buf) + ' ' + sentence
        buf.clear()

    if sentence.split().__len__() <= min:
        buf.append(sentence)
    else:
        sentences.append(sentence)

def extract_raw_sentences(soup, min):
    buf = []
    sentences = []

    for elem in soup.find_all('p'):
        text = elem.text.strip()
        if text == '' or len(text) <= 10:
            continue

        text = re.sub(r"\[\d+\]", ' ', text)

        for sentence in text.split('. '):
            parse_sentence(sentence, sentences, buf, min)

    if buf:
        buf_tokens = sum([len(sent.split()) for sent in buf])
        if buf_tokens >= min:
            sentences.append(' '.join(buf))
    
    return sentences

def clean_sentences(sentences, min, max):
    out = []
    for sentence in sentences:
        if len(sentence.split()) <= max:
            out.append(sentence)
            continue

        buf = []
        first = True
        for word in sentence.split():
            buf.append(word)
            if len(buf) >= max:
                chunk = ' '.join(buf)
                if first:
                    chunk = chunk + '...'
                else:
                    chunk = '...' + chunk
                first = False
                out.append(chunk)
                buf = []

        if len(buf) >= min:
            out.append('...' + ' '.join(buf))
    
    return out

def get_sentences(article_id, soup, min, max):
    sentences = extract_raw_sentences(soup, min)
    sentences = clean_sentences(sentences, min, max)
    timestamp = time.time()
    return [
        (timestamp, timestamp, sentence, sentence.split().__len__(), article_id)
        for sentence in sentences
    ]
  

In [ ]:
import spacy
import numpy as np

nlp = spacy.load("en_core_web_md")

text = "The dog chased the ball."
doc = nlp(text)

# Sentence/document embedding
embedding = doc.vector
print(embedding.shape)   # usually (300,)



TypeError: ForwardRef._evaluate() missing 1 required keyword-only argument: 'recursive_guard'

In [ ]:
headers = {'User-Agent': 'MyApp/1.0 (you@example.com)'}
url = f'https://en.wikipedia.org/w/rest.php/v1/page/{article_id}/html'
r = requests.get(url, headers=headers)
r.raise_for_status()
t = r.text
soup = BeautifulSoup(t)
sentences = get_sentences(article_id, soup, MIN_SENTENCE_WORDS, MAX_SENTENCE_WORDS)
embeddings = [nlp(x[2]) for x in sentences]
print(len(embeddings))

In [ ]:
# print(sentences)

[(1777941148.708642, 1777941148.708642, 'Harry Potter is a series of seven fantasy novels written by British author J.', 14, 'Harry_Potter'), (1777941148.708642, 1777941148.708642, 'K. Rowling. The novels chronicle the lives of a young wizard, Harry Potter, and his friends, Ron Weasley and Hermione Granger, all of whom are students at Hogwarts School of Witchcraft and Wizardry.', 33, 'Harry_Potter'), (1777941148.708642, 1777941148.708642, "The main story arc concerns Harry's conflict with Lord Voldemort, a dark wizard who intends to become immortal, overthrow the wizard governing body known as the Ministry of Magic, and subjugate all wizards and non-magical people, known in-universe as Muggles.", 40, 'Harry_Potter'), (1777941148.708642, 1777941148.708642, 'The series was originally published in English by Bloomsbury in the United Kingdom and Scholastic Press in the United States.', 20, 'Harry_Potter'), (1777941148.708642, 1777941148.708642, 'A series of many genres, including fantasy, 

In [ ]:
# embeddings = list(
#     model.embed(
#         [x[2] for x in sentences] 
#     )
# )
